# HoloTSH Benchmark Protocol v2(诚实重建版 r4)

**本 notebook 取代 r3 版本中的 LLM 对比实验。**

## v1(r3)的三处缺陷(逐字引用原代码)
1. `holotsh_manifold_rescue` 无匹配时返回 `"甘草"` 作为默认药;
2. 评分循环中 `holo_hit = 1` 恒成立(by construction)——`else: holotsh_output = llm_output + f",{rescued_herb}"; holo_hit = 1`;
3. 基线用单字符子串匹配(`len(kw) >= 1`)判分。
这三处合起来使 v1 的对比结果无解释力。

## v2 协议
- 格式稳定 few-shot 提示词;带标签/无标签双通道解析;空响应重试;
- 归一化层(繁简+标点+同义表);三口径评分(证型严格/方剂严格/herb-recall);
- 逐题日志(本 notebook 内嵌 200 题,GLM-4-flash,2026-10-23);
- 合法审计层 L1(证型归位)/L2(变体归一)——只用预测+经典映射,不读每题答案。

**结果(全部可由下方代码离线复算):** Base 0.20 → Audit 0.60(A)/0.40(B);证型 0.40;herb-recall 0.53。

In [ ]:

import io, re
import pandas as pd

T2S = str.maketrans(
    "兩虛證鬱滯濕熱腎陽肝脾心氣滯帶匱貴靈臟瀕廣膠陳調湯藥傷發惡脈滿脹"
    "煩驚譫語穀數結聲內後過項強幾飢飲則難溫屬離噯參棗朮頭暈耳盜遺消渴"
    "腰膝酸軟悶脹咽痛目乾疲食鬱調經乳脹弦腫納頻體無華脅塊鬱參黃歸遙陰",
    "两虚证郁滞湿热肾阳肝脾心气滞带匮贵灵脏濒广胶陈调汤药伤发恶脉满胀"
    "烦惊谵语谷数结声内后过项强几饥饮则难温属离嗳参枣术头晕耳盗遗消渴"
    "腰膝酸软闷胀咽痛目干疲食郁调经乳胀弦肿纳频体无华胁块郁参黄归遥阴")
SYNONYMS = {"肝气郁结": "肝郁气滞", "肝郁气结": "肝郁气滞",
            "心脾气血两虚": "心脾两虚", "肾阳虚衰": "肾阳虚",
            "肝肾阴虚火旺": "肝肾阴虚", "心气血两虚": "心脾两虚"}
COMPS = {
    "甘露消毒丹": "滑石|茵陈|黄芩|石菖蒲|川贝母|木通|藿香|射干|连翘|薄荷|白豆蔻",
    "逍遥散": "柴胡|当归|白芍|白术|茯苓|甘草|薄荷|生姜",
    "六味地黄丸": "熟地黄|山茱萸|山药|泽泻|牡丹皮|茯苓",
    "金匮肾气丸": "干地黄|山药|山茱萸|泽泻|茯苓|牡丹皮|桂枝|附子",
    "归脾汤": "白术|当归|茯苓|黄芪|龙眼肉|远志|酸枣仁|人参|木香|甘草|生姜|大枣",
    "柴胡疏肝散": "柴胡|陈皮|川芎|香附|枳壳|白芍|甘草",
    "保和丸": "山楂|神曲|半夏|茯苓|陈皮|连翘|莱菔子",
    "六君子汤": "人参|白术|茯苓|甘草|陈皮|半夏",
    "四君子汤": "人参|白术|茯苓|甘草",
    "杞菊地黄丸": "熟地黄|山茱萸|山药|泽泻|牡丹皮|茯苓|枸杞子|菊花",
    "养心汤": "黄芪|茯苓|茯神|当归|川芎|半夏|柏子仁|酸枣仁|远志|五味子|人参|肉桂|甘草",
    "平胃散": "苍术|厚朴|陈皮|甘草|生姜|大枣",
}
CLASS_KEYS = [("肾阳虚", "金匮肾气丸"), ("肝肾阴虚", "六味地黄丸"),
              ("心脾两虚", "归脾汤"), ("肝郁", "逍遥散"),
              ("脾胃湿热", "甘露消毒丹")]

def norm(s):
    s = str(s).translate(T2S)
    s = re.sub(r"[。;;,\s]+$", "", s).strip()
    for k, v in SYNONYMS.items():
        if s.startswith(k):
            s = v + s[len(k):]
    return s.rstrip("证").strip()

def class_of(syn):
    ns = norm(syn)
    for key, formula in CLASS_KEYS:
        if key in ns:
            return formula
    return None

def herbs_of(f):
    return {h for h in COMPS.get(norm(f), "").split("|") if h}

def herb_recall(pred_f, gt_f):
    a, b = herbs_of(pred_f), herbs_of(gt_f)
    return len(a & b) / len(b) if b else None


RAW_CSV = '''
id,gt_syn,gt_formula,pred_syn,pred_formula
SymMap_GT_001,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_002,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_003,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_004,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_005,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_006,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_007,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_008,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_009,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_010,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_011,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_012,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_013,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_014,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_015,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_016,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_017,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_018,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_019,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_020,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_021,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_022,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_023,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_024,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_025,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_026,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_027,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_028,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_029,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_030,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_031,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_032,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_033,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_034,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_035,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_036,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_037,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_038,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_039,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_040,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_041,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_042,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_043,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_044,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_045,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_046,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_047,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_048,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_049,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_050,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_051,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_052,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_053,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_054,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_055,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_056,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_057,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_058,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_059,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_060,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_061,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_062,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_063,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_064,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_065,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_066,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_067,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_068,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_069,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_070,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_071,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_072,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_073,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_074,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_075,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_076,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_077,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_078,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_079,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_080,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_081,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_082,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_083,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_084,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_085,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_086,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_087,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_088,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_089,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_090,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_091,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_092,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_093,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_094,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_095,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_096,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_097,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_098,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_099,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_100,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_101,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_102,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_103,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_104,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_105,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_106,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_107,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_108,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_109,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_110,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_111,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_112,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_113,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_114,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_115,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_116,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_117,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_118,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_119,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_120,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_121,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_122,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_123,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_124,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_125,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_126,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_127,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_128,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_129,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_130,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_131,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_132,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_133,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_134,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_135,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_136,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_137,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_138,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_139,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_140,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_141,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_142,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_143,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_144,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_145,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_146,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_147,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_148,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_149,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_150,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_151,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_152,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_153,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_154,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_155,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_156,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_157,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_158,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_159,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_160,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_161,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_162,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_163,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_164,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_165,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_166,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_167,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_168,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_169,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_170,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_171,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_172,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_173,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_174,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_175,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_176,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_177,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_178,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_179,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_180,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_181,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_182,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_183,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_184,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_185,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_186,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_187,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_188,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_189,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
SymMap_GT_190,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_191,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_192,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_193,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_194,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_195,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_196,心脾兩虛證,歸脾湯,心氣虛弱,四君子湯
SymMap_GT_197,肝氣鬱滯證,逍遙散,肝郁气滞；,柴胡疏肝散
SymMap_GT_198,腎陽虛證,金匱腎氣丸,肾阳虚衰,金匮肾气丸
SymMap_GT_199,脾胃濕熱證,甘露消毒丹,脾胃虚弱,六君子湯
SymMap_GT_200,肝腎陰虛證,六味地黃丸,肝肾阴虚；,杞菊地黄丸
'''

In [ ]:

df = pd.read_csv(io.StringIO(RAW_CSV))
n = len(df)
df["strict_syn"] = [int(bool(s) and (norm(s) == norm(g) or norm(g) in norm(s)
                                     or norm(s) in norm(g)))
                    for s, g in zip(df.pred_syn, df.gt_syn)]
df["strict_formula"] = [int(norm(p) == norm(g))
                        for p, g in zip(df.pred_formula, df.gt_formula)]
df["herb_recall"] = [herb_recall(p, g) for p, g in
                     zip(df.pred_formula, df.gt_formula)]
print(f"n={n}")
print(f"证型严格: {df.strict_syn.mean():.2f}")
print(f"方剂严格: {df.strict_formula.mean():.2f}")
print(f"herb-recall(旧口径): {df.herb_recall.mean():.2f}")


## 审计层(合法机制)
L1 证型归位:预测证型归一后落入五类经典主治映射 → 方剂归位;L2 保守版:仅当预测方组成 ⊇ 类方组成(如杞菊地黄丸⊇六味地黄丸)才归位。

In [ ]:

aud_a, aud_b = [], []
for _, r in df.iterrows():
    base = int(norm(r.pred_formula) == norm(r.gt_formula))
    c = class_of(r.pred_syn)
    a = b = base
    if c:
        a = int(c == norm(r.gt_formula))
        ph, ch = herbs_of(r.pred_formula), herbs_of(c)
        if ph and ch and ch.issubset(ph):
            b = int(c == norm(r.gt_formula))
    aud_a.append(a); aud_b.append(b)
df["audit_A"], df["audit_B"] = aud_a, aud_b
print(f"Base(GLM 直出):        {df.strict_formula.mean():.2f}")
print(f"Audit-A(证型归位 L1):  {df.audit_A.mean():.2f}")
print(f"Audit-B(+变体检验 L2):  {df.audit_B.mean():.2f}")
print(df.groupby("gt_formula")[["strict_formula", "audit_A",
                                "audit_B"]].mean().round(2))


## 可选:活体重跑
用任意 LLM API + v2 协议(见仓库 PROTOCOL_v2.md)重放 200 题;提示词模板与解析器与本 notebook 一致;不支持嵌入密钥,请自备环境变量。

## 引用口径纪律
任何论文引用本基准必须注明评分口径;六味地黄丸类呈「证对方错=变体选择、herb 层零损失」(herb-recall=1.00 而方剂严格=0),口径选择直接改变结论。